In [1]:
# ============================================================
# LOAN RISK PREDICTION - FINAL MODEL
# ============================================================

# 1. IMPORT LIBRARIES
# pandas       → Used for working with tabular data
# joblib       → Used to save the trained ML model
# sklearn      → Used for machine learning

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ============================================================
# 2. LOAD DATASET
# ============================================================

# Read the CSV file into a Pandas DataFrame.
# Our full dataset contains 614 rows and 13 columns.

data = pd.read_csv("../data/full_loan_data.csv")

print("Dataset shape:", data.shape)


# ============================================================
# 3. REMOVE UNNECESSARY COLUMN
# ============================================================

# Loan_ID is only an identifier.
# It does not help the model predict loan approval.

data = data.drop("Loan_ID", axis=1)


# ============================================================
# 4. DEFINE COLUMNS
# ============================================================

# Categorical columns contain text/categories.

categorical_columns = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area"
]


# Numerical columns contain numbers.

numerical_columns = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]


# ============================================================
# 5. HANDLE MISSING VALUES
# ============================================================

# For categorical columns:
# Replace missing values with the most common value (mode).

for column in categorical_columns:
    data[column] = data[column].fillna(
        data[column].mode()[0]
    )


# For numerical columns:
# Replace missing values with the median value.

for column in numerical_columns:
    data[column] = data[column].fillna(
        data[column].median()
    )


# ============================================================
# 6. SEPARATE FEATURES (X) AND TARGET (y)
# ============================================================

# X = information used to make the prediction
# y = answer we want the model to learn

X = data.drop("Loan_Status", axis=1)
y = data["Loan_Status"]

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 7. CREATE PREPROCESSOR
# ============================================================

# Machine learning models cannot directly understand
# text such as "Male", "Graduate", or "Urban".
#
# OneHotEncoder converts categorical values into numbers.
#
# remainder="passthrough" keeps the numerical columns unchanged.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        )
    ],
    remainder="passthrough"
)


# ============================================================
# 8. SPLIT DATA INTO TRAINING AND TESTING DATA
# ============================================================

# 80% → training
# 20% → testing
#
# random_state=42 → keeps the same split every time.
# stratify=y → keeps the Y/N distribution balanced.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


# ============================================================
# 9. CREATE MACHINE LEARNING PIPELINE
# ============================================================

# Pipeline connects:
#
# Raw data
#    ↓
# Preprocessor
#    ↓
# Logistic Regression
#
# This means preprocessing and prediction happen together.

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])


# ============================================================
# 10. TRAIN THE MODEL
# ============================================================

# fit() teaches the model using the training data.

pipeline.fit(X_train, y_train)

print("Model training completed!")


# ============================================================
# 11. MAKE PREDICTIONS
# ============================================================

# predict() asks the trained model to predict
# the loan status of the test applicants.

predictions = pipeline.predict(X_test)

print("\nFirst 10 predictions:")
print(predictions[:10])


# ============================================================
# 12. CALCULATE ACCURACY
# ============================================================

# Accuracy = correct predictions / total predictions

accuracy = accuracy_score(y_test, predictions)

print("\nAccuracy:", accuracy)


# ============================================================
# 13. CONFUSION MATRIX
# ============================================================

# Shows:
# Correct N predictions
# Incorrect N predictions
# Incorrect Y predictions
# Correct Y predictions

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))


# ============================================================
# 14. CLASSIFICATION REPORT
# ============================================================

# Shows:
# Precision
# Recall
# F1-score
# Support

print("\nClassification Report:")
print(classification_report(y_test, predictions))


# ============================================================
# 15. SAVE THE COMPLETE PIPELINE
# ============================================================

# Save preprocessing + trained model together.
#
# This file will later be used by our Streamlit app.

joblib.dump(
    pipeline,
    "../models/loan_pipeline.pkl"
)

print("\nFinal model saved successfully!")


# ============================================================
# 16. TEST WITH A NEW APPLICANT
# ============================================================

# Create information for a completely new applicant.

new_applicant = pd.DataFrame({
    "Gender": ["Male"],
    "Married": ["Yes"],
    "Dependents": ["0"],
    "Education": ["Graduate"],
    "Self_Employed": ["No"],
    "ApplicantIncome": [5000],
    "CoapplicantIncome": [2000],
    "LoanAmount": [150],
    "Loan_Amount_Term": [360],
    "Credit_History": [1],
    "Property_Area": ["Urban"]
})


# Ask the trained pipeline to predict the loan status.

new_prediction = pipeline.predict(new_applicant)

print("\nNew Applicant Prediction:", new_prediction[0])


# ============================================================
# 17. PREDICTION PROBABILITY
# ============================================================

# predict_proba() gives the probability for each class.

probability = pipeline.predict_proba(new_applicant)

print("\nPrediction Probabilities:")
print(probability)

print("\nClasses:")
print(pipeline.classes_)

Dataset shape: (614, 13)
X shape: (614, 11)
y shape: (614,)
Training rows: 491
Testing rows: 123
Model training completed!

First 10 predictions:
['N' 'Y' 'Y' 'Y' 'Y' 'Y' 'Y' 'Y' 'Y' 'Y']

Accuracy: 0.8292682926829268

Confusion Matrix:
[[23 15]
 [ 6 79]]

Classification Report:
              precision    recall  f1-score   support

           N       0.79      0.61      0.69        38
           Y       0.84      0.93      0.88        85

    accuracy                           0.83       123
   macro avg       0.82      0.77      0.78       123
weighted avg       0.83      0.83      0.82       123


Final model saved successfully!

New Applicant Prediction: Y

Prediction Probabilities:
[[0.21748617 0.78251383]]

Classes:
['N' 'Y']
